### Import relevant libraries

In [1]:
import pandas as pd
import numpy as np

### Load dataset

In [2]:
image_encoder_type = 'clip'  # 'clip' or 'vit'
image_encoder_type

'clip'

In [3]:
if image_encoder_type == 'clip':
    # Load the data extracted from video frames using CLIP pretrained model
    df_action = pd.read_csv('../datasets/video_frame_features_clip.csv')
elif image_encoder_type == 'vit':
    # Load the data extracted from video frames using ViT pretrained model
    df_action = pd.concat([pd.read_csv(f'../datasets/video_frame_features_vit_part_{part}.csv') for part in range(7)], ignore_index=True)


In [4]:
df_action.head()

,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,...,feature_506,feature_507,feature_508,feature_509,feature_510,feature_511,feature_512,video_file_name,label,subset
0,0.006786,-0.3203,0.3706,-0.006176,-0.1317,-0.1846,0.002857,-0.09674,-0.1760,-0.1564,...,-0.04333,-0.2288,0.10486,0.017010,0.2267,0.1471,0.2890,uccrime_Robbery126_x264_trimmed.mp4,abnormal,train
1,-0.009770,-0.3535,0.3987,-0.007480,-0.1360,-0.2051,0.008484,-0.08820,-0.2448,-0.1943,...,-0.10205,-0.2418,0.13650,0.000902,0.1506,0.1405,0.2947,uccrime_Robbery126_x264_trimmed.mp4,abnormal,train
2,0.014490,-0.3508,0.3920,0.002000,-0.1412,-0.1996,0.018860,-0.06880,-0.3167,-0.1968,...,-0.10406,-0.2332,0.13270,0.000527,0.2075,0.1459,0.2861,uccrime_Robbery126_x264_trimmed.mp4,abnormal,train
3,0.013010,-0.3516,0.3909,-0.000943,-0.1395,-0.2040,0.018390,-0.07294,-0.3164,-0.1943,...,-0.10190,-0.2317,0.13220,-0.001555,0.2031,0.1480,0.2852,uccrime_Robbery126_x264_trimmed.mp4,abnormal,train
4,-0.007187,-0.3738,0.4062,-0.004604,-0.1440,-0.2162,0.007750,-0.05478,-0.3389,-0.2120,...,-0.07660,-0.1879,0.12415,-0.010605,0.1663,0.1577,0.2793,uccrime_Robbery126_x264_trimmed.mp4,abnormal,train


In [39]:
df_action.tail()

,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,...,feature_506,feature_507,feature_508,feature_509,feature_510,feature_511,feature_512,video_file_name,label,subset
1324153,-0.1649,0.2036,0.5790,0.000816,0.3230,-0.088100,-0.15640,-0.03296,-0.10693,-0.05496,...,-0.1560,-0.11070,-0.09375,0.6930,0.3416,-0.2832,-0.2632,casia_topdownview_p24_faint_a1_augmented.mp4,abnormal,train
1324154,-0.1700,0.2267,0.5654,-0.044980,0.3345,-0.064700,-0.10540,-0.00592,-0.14720,-0.05270,...,-0.1648,-0.05914,-0.06690,0.6890,0.2764,-0.2870,-0.3572,casia_topdownview_p24_faint_a1_augmented.mp4,abnormal,train
1324155,-0.1932,0.1733,0.5510,-0.020320,0.3410,-0.011765,-0.08044,-0.02742,-0.10600,-0.03375,...,-0.1455,-0.06396,-0.06250,0.7344,0.2720,-0.2908,-0.3003,casia_topdownview_p24_faint_a1_augmented.mp4,abnormal,train
1324156,-0.1954,0.1663,0.5615,-0.024030,0.3462,-0.015396,-0.08270,-0.00932,-0.11597,-0.02513,...,-0.1448,-0.07385,-0.04843,0.7383,0.2744,-0.2854,-0.3022,casia_topdownview_p24_faint_a1_augmented.mp4,abnormal,train
1324157,-0.2019,0.1952,0.5440,-0.032750,0.3447,-0.007572,-0.08530,0.00659,-0.11163,-0.04570,...,-0.1455,-0.05280,-0.05615,0.7360,0.2869,-0.2827,-0.3050,casia_topdownview_p24_faint_a1_augmented.mp4,abnormal,train


In [5]:
df_action.shape

(1324158, 515)

### Explore Data

In [6]:
# Check for missing values
df_action.isnull().sum()

feature_1          0
feature_2          0
feature_3          0
feature_4          0
feature_5          0
                  ..
feature_511        0
feature_512        0
video_file_name    0
label              0
subset             0
Length: 515, dtype: int64

In [7]:
# Check for label distribution
df_action['label'].value_counts()

label
abnormal    1051114
normal       273044
Name: count, dtype: int64

In [8]:
# Check number of unique video files per label
df_action.groupby('label')['video_file_name'].nunique()

label
abnormal    1079
normal      1070
Name: video_file_name, dtype: int64

In [9]:
# Check number of unieque video files
df_action['video_file_name'].nunique()

2149

In [10]:
# Check number of labels by subset
df_action.groupby('subset')['label'].value_counts()

subset  label   
test    abnormal    168217
        normal       40561
train   abnormal    674931
        normal      187512
val     abnormal    207966
        normal       44971
Name: count, dtype: int64

In [11]:
# Check number of unique video files per subset
df_action.groupby('subset')['video_file_name'].nunique()

subset
test      327
train    1496
val       326
Name: video_file_name, dtype: int64

In [12]:
df_action['subset'].unique()

array(['train', 'val', 'test'], dtype=object)

### Data preparation

In [13]:
SEED = 42
np.random.seed(SEED)

In [14]:
# Helper function to convert series to supervised learning
def series_to_supervised(data, column_names, n_in=1, n_out=1, dropnan=True, n_out_index=None, is_drop_target=False, lag_sampling=1):
    """
    convert series to supervised learning
    Params:
    - n_in: number of timesteps
    - n_out: number of output timestep for prediction | eg. 1 day, 2 days, ....
    - n_out_index: define specific variable for prediction
    - is_drop_target: drop target variable from input variables (to be used for classification problem)
    - step_size: step size for selecting input sequence timesteps
    """
    df_new = pd.DataFrame(data)
    
    if is_drop_target:
        df_input = df_new.drop(columns=[column_names[n_out_index]])
    else:
        df_input = df_new.copy()
    
    n_vars = df_input.shape[1]
    
    cols, names = list(), list()
    
    # input sequence (t-n, ... t-1)
    for i in range(n_in, 0, -lag_sampling):
        cols.append(df_input.shift(i))
        names += [f'{column_names[j]}(t-{i})' for j in range(n_vars)]
    
    # forecast sequence (t, t+1, ..., t+n)
    for i in range(0, n_out):
        if n_out_index:
            cols.append(df_new.shift(-i).iloc[:, n_out_index])
            if i == 0:
                names += [f'{column_names[n_out_index]}(t)']
            else:
                names += [f'{column_names[n_out_index]}(t+{i})']
        else:
            cols.append(df_new.shift(-i))
            if i == 0:
                names += [f'{column_names[j]}(t)' for j in range(n_vars)]
            else:
                names += [f'{column_names[j]}(t+{i})' for j in range(n_vars)]
    # put it all together
    agg = pd.concat(cols, axis=1)
    agg.columns = names
    # drop rows with NaN values that generated from df_new.shift
    if dropnan:
        agg.dropna(inplace=True)
    
    return agg

In [15]:
# define features
column_names = df_action.columns.tolist()
column_names.remove('video_file_name')
column_names.remove('subset')
column_names

['feature_1',
 'feature_2',
 'feature_3',
 'feature_4',
 'feature_5',
 'feature_6',
 'feature_7',
 'feature_8',
 'feature_9',
 'feature_10',
 'feature_11',
 'feature_12',
 'feature_13',
 'feature_14',
 'feature_15',
 'feature_16',
 'feature_17',
 'feature_18',
 'feature_19',
 'feature_20',
 'feature_21',
 'feature_22',
 'feature_23',
 'feature_24',
 'feature_25',
 'feature_26',
 'feature_27',
 'feature_28',
 'feature_29',
 'feature_30',
 'feature_31',
 'feature_32',
 'feature_33',
 'feature_34',
 'feature_35',
 'feature_36',
 'feature_37',
 'feature_38',
 'feature_39',
 'feature_40',
 'feature_41',
 'feature_42',
 'feature_43',
 'feature_44',
 'feature_45',
 'feature_46',
 'feature_47',
 'feature_48',
 'feature_49',
 'feature_50',
 'feature_51',
 'feature_52',
 'feature_53',
 'feature_54',
 'feature_55',
 'feature_56',
 'feature_57',
 'feature_58',
 'feature_59',
 'feature_60',
 'feature_61',
 'feature_62',
 'feature_63',
 'feature_64',
 'feature_65',
 'feature_66',
 'feature_67',
 'fe

In [45]:
# Define window_size (number of timesteps or number of frames), number of features, and number of output step
window_size = 120 # 120 frames ~ 4 seconds
lag_sampling = 15 # step size for selecting frames within a window
num_features = len(column_names[:-1])
n_steps_out = 1
n_out_index =  -1

In [ ]:
# Reframe as supervised learning by each video file
subsets = df_action['subset'].unique()
df_by_subset = {}

for subset in subsets:
    print(f'Processing subset: {subset}')
    df_action_subset = df_action[df_action['subset'] == subset]
    video_files = df_action_subset['video_file_name'].unique()

    dfs = []
    n_skip = 0
    for i, video_file in enumerate(video_files):
        print(f'Processing video file {i+1}/{len(video_files)}: {video_file}'.ljust(200), end='\r')
        df_video = df_action_subset[df_action_subset['video_file_name'] == video_file]
        if df_video.shape[0] < int(window_size/lag_sampling):
            n_skip += 1
            continue
        elif df_video.shape[0] < window_size:
            idx = np.linspace(0, df_video.shape[0]-1, num=int(window_size/lag_sampling), dtype=int)
            df_video = df_video.iloc[idx]

        df_reframed = series_to_supervised(
            df_video.drop(columns=['video_file_name', 'subset']),
            column_names,
            n_in=window_size,
            n_out=n_steps_out,
            n_out_index=n_out_index,
            is_drop_target=True,
            lag_sampling=lag_sampling
        )
        dfs.append(df_reframed)

    df_subset = pd.concat(dfs, ignore_index=True)
    df_by_subset[subset] = df_subset
    print(f"Finished processing subset: {subset}. Skipped {n_skip} videos.".ljust(200))


Processing subset: train
Finished processing subset: train. Skipped 1 videos.                                                                                                                                                    
Processing subset: val
Finished processing subset: val. Skipped 1 videos.                                                                                                                                                      
Processing subset: test
Finished processing subset: test. Skipped 0 videos.                                                                                                                                                     


In [18]:
# Check dataframes size for each subset
for key in df_by_subset.keys():
    print(f'Subset: {key}, Shape: {df_by_subset[key].shape}')

Subset: train, Shape: (695395, 4097)
Subset: val, Shape: (216694, 4097)
Subset: test, Shape: (171769, 4097)


In [19]:
# Rename label column
for key in df_by_subset.keys():
    df_by_subset[key].rename(columns={f'label(t)': 'label'}, inplace=True)

In [20]:
# Check label distribution for each subset
for key in df_by_subset.keys():
    print(f'Subset: {key} \n{df_by_subset[key]["label"].value_counts()}')

Subset: train 
label
abnormal    589601
normal      105794
Name: count, dtype: int64
Subset: val 
label
abnormal    189140
normal       27554
Name: count, dtype: int64
Subset: test 
label
abnormal    149417
normal       22352
Name: count, dtype: int64


In [31]:
# Define sample size for each subset
# train: 70%, val: 15%, test: 15%
sample_size = {
    'train': 160000,
    'val': 20000,
    'test': 20000
}

In [32]:
# Sample data for each subset
df_train = pd.concat([df_by_subset['train'][df_by_subset['train']['label'] == label].sample(n=int(sample_size['train']/2), random_state=SEED) for label in df_by_subset['train']['label'].unique()])
df_train.shape

(160000, 4097)

In [33]:
df_val = pd.concat([df_by_subset['val'][df_by_subset['val']['label'] == label].sample(n=int(sample_size['val']/2), random_state=SEED) for label in df_by_subset['val']['label'].unique()])
df_val.shape

(20000, 4097)

In [34]:
df_test = pd.concat([df_by_subset['test'][df_by_subset['test']['label'] == label].sample(n=int(sample_size['test']/2), random_state=SEED) for label in df_by_subset['test']['label'].unique()])
df_test.shape

(20000, 4097)

In [35]:
# Check label distribution after sampling
print(f'Train label distribution:\n{df_train["label"].value_counts()}') 
print(f'Validation label distribution:\n{df_val["label"].value_counts()}')
print(f'Test label distribution:\n{df_test["label"].value_counts()}')

Train label distribution:
label
abnormal    80000
normal      80000
Name: count, dtype: int64
Validation label distribution:
label
abnormal    10000
normal      10000
Name: count, dtype: int64
Test label distribution:
label
abnormal    10000
normal      10000
Name: count, dtype: int64


In [36]:
# Convert labels to binary (0: normal, 1: abnormal)
df_train['label'] = df_train['label'].map(lambda x: 0 if x == 'abnormal' else 1)
df_val['label'] = df_val['label'].map(lambda x: 0 if x == 'abnormal' else 1)
df_test['label'] = df_test['label'].map(lambda x: 0 if x == 'abnormal' else 1)

In [41]:
df_train.head()

,feature_1(t-120),feature_2(t-120),feature_3(t-120),feature_4(t-120),feature_5(t-120),feature_6(t-120),feature_7(t-120),feature_8(t-120),feature_9(t-120),feature_10(t-120),...,feature_504(t-15),feature_505(t-15),feature_506(t-15),feature_507(t-15),feature_508(t-15),feature_509(t-15),feature_510(t-15),feature_511(t-15),feature_512(t-15),label
606959,0.1276,-0.19810,0.03925,-0.4060,-0.01877,0.38090,0.10394,0.06805,-0.04010,0.16760,...,-0.014145,0.33280,-0.20080,0.26420,0.02713,0.11000,0.44850,0.12110,-0.06604,0
258689,-0.2241,0.21340,0.53600,-0.3667,0.21920,-0.58450,-0.21450,-0.19170,0.08746,-0.01506,...,-0.194300,-0.26370,-0.11550,-0.03186,0.09735,0.39300,0.07733,-0.13070,0.23120,0
209397,0.1449,0.05746,0.22660,-0.1960,0.08550,-0.19360,0.27080,-0.31150,-0.31370,-0.01073,...,-0.116000,0.28270,-0.09546,0.11896,0.18420,0.08887,0.47900,-0.04852,0.11360,0
240063,0.3293,-0.05240,0.19520,-0.3276,-0.04642,-0.05823,0.11830,-0.10170,-0.01976,-0.01677,...,-0.205600,0.01634,-0.26500,0.06042,0.20680,0.01587,0.73500,-0.40400,0.05402,0
399403,0.1425,-0.30760,0.21200,-0.1345,-0.24350,-0.24910,0.25730,-0.13110,-0.04025,0.13660,...,-0.335700,0.17200,-0.16000,-0.15770,0.21180,-0.19930,0.08620,0.00046,0.40260,0


In [42]:
df_train.tail()

,feature_1(t-120),feature_2(t-120),feature_3(t-120),feature_4(t-120),feature_5(t-120),feature_6(t-120),feature_7(t-120),feature_8(t-120),feature_9(t-120),feature_10(t-120),...,feature_504(t-15),feature_505(t-15),feature_506(t-15),feature_507(t-15),feature_508(t-15),feature_509(t-15),feature_510(t-15),feature_511(t-15),feature_512(t-15),label
464016,0.15230,0.151500,-0.01776,-0.38800,0.2790,-0.03430,0.11900,0.66700,0.2817,0.22630,...,-0.012634,-0.272200,-0.6230,0.2002,0.05743,-0.1888,1.309,0.0186,-0.08276,1
510665,0.01517,0.189700,0.26540,-0.39260,0.6070,-0.15120,0.27760,0.10126,0.1537,-0.04108,...,0.100160,-0.493400,-0.0788,-0.2269,0.11670,0.4810,0.635,-0.5615,-0.64840,1
441453,0.08240,0.012024,0.32800,-0.05884,0.4265,0.23600,0.10670,0.11220,0.2850,-0.22670,...,-0.183700,0.079830,-0.1986,0.1619,0.19290,0.3710,0.763,-0.3470,-0.30740,1
468891,0.06903,0.136600,0.29470,-0.03160,0.2666,0.30350,0.29370,0.12740,0.0426,-0.26000,...,-0.286400,-0.004406,-0.3042,0.1438,0.15690,0.3638,0.734,-0.1950,-0.15420,1
521968,0.11140,0.118300,0.20150,-0.12054,0.3115,0.07635,0.07336,0.09735,0.0768,-0.22020,...,-0.074460,-0.060670,-0.2095,0.1136,0.09480,0.3838,0.765,-0.2847,-0.33600,1


In [46]:
# Reshape data for model training
X_train, y_train = df_train.drop(columns=['label']).values, df_train['label'].values
X_val, y_val = df_val.drop(columns=['label']).values, df_val['label'].values
X_test, y_test = df_test.drop(columns=['label']).values, df_test['label'].values

effective_window_size = int(window_size / lag_sampling)
X_train = X_train.reshape((-1, effective_window_size, num_features))
X_val = X_val.reshape((-1, effective_window_size, num_features))
X_test = X_test.reshape((-1, effective_window_size, num_features))

X_train.shape, y_train.shape, X_val.shape, y_val.shape, X_test.shape, y_test.shape

((160000, 8, 512),
 (160000,),
 (20000, 8, 512),
 (20000,),
 (20000, 8, 512),
 (20000,))

In [ ]:
# Save processed data to csv files
np.savez_compressed(f'../datasets/video_frame_features_{image_encoder_type}_ews{effective_window_size}_supervised_data.npz', 
                    X_train=X_train, y_train=y_train, 
                    X_val=X_val, y_val=y_val, 
                    X_test=X_test, y_test=y_test)